In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Gaussian Naive Bayes Triage Classifier (`models/gauss_nb.ipynb`)

This notebook trains a **Gaussian Naive Bayes Classifier** (`e1071::naiveBayes`) for ESI triage level prediction using feature-engineered continuous & numeric indicators:
- **Input Features**:
  - `age` (continuous patient age)
  - `gender` (0 = Female, 1 = Male)
  - `cc_breathingdifficulty` (Chief Complaint flag for breathing difficulty)
  - **10 Clinical Feature Engineered Flags** defined in `TODO.md`: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
- **Model Framework**: Gaussian Naive Bayes (`e1071::naiveBayes`) fitting Gaussian likelihood densities $\mathcal{N}(\mu_{i,c}, \sigma_{i,c}^2)$ for numeric predictors.
- **Evaluation Metrics**: **Accuracy**, **Multi-Class ROC-AUC**, **Log Loss**, and **Per-Class Precision / Recall / F1**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(e1071)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct Gaussian Feature Set
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col   <- config$classes$target_col
target_classes <- as.character(config$classes$outputs)

# Extract / compute gender
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0

# Extract / compute cc_breathingdifficulty chief complaint
cc_breathing <- if ("cc_breathingdifficulty" %in% names(raw_df)) {
  ifelse(raw_df$cc_breathingdifficulty == 1, 1, 0)
} else if ("cc_shortnessofbreath" %in% names(raw_df)) {
  ifelse(raw_df$cc_shortnessofbreath == 1, 1, 0)
} else {
  rep(0, nrow(raw_df))
}

# Construct Gaussian Feature Dataframe (numeric continuous/indicator columns)
df_gnb <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_breathing,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

df_gnb[[target_col]] <- factor(as.character(raw_df[[target_col]]), levels = target_classes)

if (any(is.na(df_gnb))) {
  df_gnb <- na.omit(df_gnb)
}

cat(sprintf("Gaussian Naive Bayes Dataset Ready: %d rows x %d cols\n", nrow(df_gnb), ncol(df_gnb)))
cat("Numeric Gaussian Predictors (13):", paste(setdiff(names(df_gnb), target_col), collapse = ", "), "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Age Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val <- createDataPartition(df_gnb[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df_gnb[in_train_val, ]
test_df      <- df_gnb[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Gaussian Naive Bayes Classifier
# ---------------------------------------------------------
set.seed(config$training$random_state)

formula_nb <- as.formula(paste(target_col, "~ ."))

cat("Training Gaussian Naive Bayes classifier (e1071::naiveBayes)...\n")
gauss_nb_model <- naiveBayes(formula_nb, data = train_df)

cat("Gaussian Naive Bayes training complete!\n")
print(gauss_nb_model)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Scoring Metrics (Accuracy, ROC-AUC, Precision, Log Loss)
# ---------------------------------------------------------
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_gauss_nb <- function(model, data, set_name, target_col) {
  prob_matrix <- predict(model, newdata = data, type = "raw")
  pred_factor <- predict(model, newdata = data, type = "class")
  actual_factor <- factor(as.character(data[[target_col]]), levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   GAUSSIAN NAIVE BAYES MODEL - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy            : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Multi-Class ROC-AUC : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss            : %.4f\n", log_loss))
  cat("\nPer-Class Precision / Recall Metrics:\n")
  print(cm$byClass[, c("Precision", "Recall", "F1")])
  cat("\nConfusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Benchmark on Validation and Test Sets
evaluate_gauss_nb(gauss_nb_model, val_df, "Validation", target_col)
evaluate_gauss_nb(gauss_nb_model, test_df, "Test", target_col)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Gaussian Naive Bayes Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "gauss_nb_model.rds")
saveRDS(list(model = gauss_nb_model, preproc = preproc), file = model_path)
cat("Gaussian Naive Bayes model saved to:", model_path, "\n")